In [1]:
import glob
import os
import pandas as pd

# Get all match IDs from the offline data folder
files = glob.glob("sb_offline_data/match_*.pkl")
match_ids = [int(os.path.basename(f).split('_')[1].split('.')[0]) for f in files]

print(f"All {len(match_ids)} match IDs found.")

All 1238 match IDs found.


In [2]:
def get_score_diff(events, team, t_seconds):
    
    # < to not count the current event if it's a goal
    past_events = events[events.minute * 60 + events.second < t_seconds]
    
    if past_events.empty:
        return 0
        
    # Normal goals
    if 'shot_outcome' in past_events.columns:
        normal_goals = past_events[(past_events['type'] == 'Shot') & (past_events['shot_outcome'] == 'Goal')]
    else:
        normal_goals = pd.DataFrame()
        
    # Own goals
    own_goals = past_events[past_events['type'] == 'Own Goal Against']
    
    # Goals of affected team
    team_scored = len(normal_goals[normal_goals['team'] == team])
    team_benefited = len(own_goals[own_goals['team'] != team])
    total_team_goals = team_scored + team_benefited
    
    # Goals of opposing team
    opp_scored = len(normal_goals[normal_goals['team'] != team])
    opp_benefited = len(own_goals[own_goals['team'] == team])
    total_opp_goals = opp_scored + opp_benefited
    
    # Goal difference from the perspective of the affected team
    return total_team_goals - total_opp_goals

In [3]:
def detect_smash_and_grab(events, match_id, window_seconds=300):
    
    # Identify all goals in the match
    goals = events[(events.type == 'Shot') & (events.shot_outcome == 'Goal')].copy()
    
    smash_and_grab_list = []
    
    for _, goal in goals.iterrows():
        goal_time = goal.minute * 60 + goal.second # Time of the goal in seconds
        
        # Identify scoring and conceding teams
        scoring_team = goal.team
        all_teams = events.team.unique()
        conceding_team = [t for t in all_teams if t != scoring_team][0]
        
        # Define the window before the goal
        window_start = max(0, goal_time - window_seconds)
        window_events = events[
            (events.minute * 60 + events.second >= window_start) &
            (events.minute * 60 + events.second < goal_time) &
            (events.period == goal.period) # Ensure we stay within the same half
        ]
        
        if window_events.empty:
            continue
            
        # Calculate cumulative xG for both teams in the window
        window_shots = window_events[window_events.type == 'Shot']
        xg_conceding_team = window_shots[window_shots.team == conceding_team].shot_statsbomb_xg.sum()
        xg_scoring_team = window_shots[window_shots.team == scoring_team].shot_statsbomb_xg.sum()
        
        # Calculate Possession Share
        # Using event count as a proxy for possession duration per team
        events_conceding = len(window_events[window_events.possession_team == conceding_team])
        total_window_events = len(window_events)
        possession_share = events_conceding / total_window_events if total_window_events > 0 else 0
        
        # Apply Smash and Grab Logic
        xg_ratio_threshold = xg_conceding_team >= (2 * xg_scoring_team)
        if possession_share >= 0.60 and xg_ratio_threshold:

            possession_diff = possession_share - (1 - possession_share)
            smash_and_grab_list.append({
                'event_id': goal.id,
                'match_id': match_id,
                'period': goal.period,
                'minute': goal.minute,
                'second': goal.second,
                'total_seconds': goal_time,
                'goal_diff': get_score_diff(events, scoring_team, goal_time),
                'event_type': 'Smash and Grab',
                'affected_team': scoring_team, # The team that scored despite of being dominated
                'severity': possession_diff, # Differential at moment of goal 
            })
            
    return smash_and_grab_list

In [4]:
def detect_converted_turnovers(events, match_id):
    
    turnover_list = []

    events['total_seconds'] = events['minute'] * 60 + events['second']
    
    # Filter for ball recoveries in the attacking third (x > 80)
    recoveries = events[
        (events.type.isin(['Ball Recovery', 'Interception'])) & 
        (events.location.apply(lambda loc: loc[0] if isinstance(loc, list) else 0) > 80)
    ].copy().sort_values(['period', 'minute', 'second'])
    
    last_pfe_time = {} 

    for _, recovery in recoveries.iterrows():
        recovery_time = recovery.total_seconds
        team = recovery.team
        period = recovery.period
        
        # Frequency filter: 5-minute window (300 seconds)
        if team in last_pfe_time and (recovery_time - last_pfe_time[team]) < 300:
            continue
        
        # Define the 15-second window after the recovery
        window_events = events[
            (events.total_seconds > recovery_time) & 
            (events.total_seconds <= recovery_time + 15) & 
            (events.period == period)
        ]
        
        # Check for any shots by the recovering team in this window
        has_shot = any(window_events[window_events.team == team].type == 'Shot')
        
        # PFE condition: SHOT by the recovering team within 15 seconds of recovery
        if has_shot:
            turnover_list.append({
                'event_id': recovery.id,
                'match_id': match_id,
                'period': recovery.period,
                'minute': recovery.minute,
                'second': recovery.second,
                'total_seconds': recovery_time,
                'goal_diff': get_score_diff(events, team, recovery_time),
                'event_type': 'Converted Turnover',
                'affected_team': team,
                'severity': recovery.location[0]
            })
            last_pfe_time[team] = recovery_time
            
    return turnover_list

In [5]:
def detect_crucial_goals(events, match_id):

    crucial_goal_list = []

    # Identify all goals in the match
    goals = events[(events.type == 'Shot') & (events.shot_outcome == 'Goal')].copy()
    
    for _, goal in goals.iterrows():
        goal_time = goal.minute * 60 + goal.second # Time of the goal in seconds
        scoring_team = goal.team # Identify the team that scored the goal
        diff_before_goal = get_score_diff(events, scoring_team, goal_time) # Get the score difference before the goal
        
        base_severity = 0
        
        if diff_before_goal == -1: # Equalising Goal
            base_severity = 1
        elif diff_before_goal == 0: # Go-Ahead Goal
            base_severity = 2
        else:
            continue

        severity = base_severity * goal.minute  # Cruciality is weighted by the minute of the match, giving more importance to goals scored later in the game.
            
        crucial_goal_list.append({
            'event_id': goal.id,
            'match_id': match_id,
            'period': goal.period,
            'minute': goal.minute,
            'second': goal.second,
            'total_seconds': goal_time,
            'goal_diff': diff_before_goal,
            'event_type': 'Crucial Goal',
            'affected_team': scoring_team,
            'severity': severity 
        })

    return crucial_goal_list

In [6]:
def detect_let_offs(events, match_id, xg_threshold=0.20):
    
    # Filter: Non-penalty shots that did not result in a goal
    potential_misses = events[
        (events.type == 'Shot') & 
        (events.shot_type != 'Penalty') & 
        (events.shot_outcome != 'Goal') &
        (events.shot_statsbomb_xg >= xg_threshold) # Minimum threshold for robustness 
    ].copy()
    
    survived_list = []
    for _, shot in potential_misses.iterrows():
        defending_team = [t for t in events.team.unique() if t != shot.team][0]
        survived_list.append({
            'event_id': shot.id,
            'match_id': match_id,
            'period': shot.period,
            'minute': shot.minute,
            'second': shot.second,
            'total_seconds': shot.minute * 60 + shot.second,
            'goal_diff': get_score_diff(events, shot.team, shot.minute * 60 + shot.second),
            'event_type': 'Let Off',
            'affected_team': defending_team, # The team that survived the big miss
            'severity': shot.shot_statsbomb_xg, # Using xG as severity
        })
    return survived_list

In [7]:
def detect_final_third_dribbles(events, match_id):

    dribbles_list = []
    
    events['total_seconds'] = events['minute'] * 60 + events['second']
    
    # Filter for completed dribbles in the final third (x >= 80)
    dribbles = events[
        (events.type == 'Dribble') &
        (events.dribble_outcome == 'Complete') &
        (events.location.apply(lambda loc: loc[0] if isinstance(loc, list) else 0) >= 80)
    ].copy().sort_values(['period', 'total_seconds'])
    
    for _, dribble in dribbles.iterrows():
        dribble_time = dribble.total_seconds
        team = dribble.team
        period = dribble.period
        
        # Define the 15-second window after the dribble
        window_events = events[
            (events.total_seconds > dribble_time) & 
            (events.total_seconds <= dribble_time + 15) & 
            (events.period == period)
        ]
        
        # PFE condition: Check if there was a shot by the dribbling team in this window
        has_shot = any(window_events[window_events.team == team].type == 'Shot')
        
        if has_shot:
            dribbles_list.append({
                'event_id': dribble.id,
                'match_id': match_id,
                'period': dribble.period,
                'minute': dribble.minute,
                'second': dribble.second,
                'total_seconds': dribble_time,
                'goal_diff': get_score_diff(events, team, dribble_time),
                'event_type': 'Final Third Dribble',
                'affected_team': team, 
                'severity': dribble.location[0]
            })
        
    return dribbles_list

In [8]:
def filter_isolated_events(pfe_df, window_seconds=300):

    if pfe_df.empty:
        return pfe_df
    
    # Sort by match and time
    df = pfe_df.sort_values(['match_id', 'total_seconds']).copy()
    
    # Time difference to the previous PFE in the same match
    df['prev_diff'] = df.groupby('match_id')['total_seconds'].diff()
    
    # Time difference to the next PFE in the same match
    df['next_diff'] = df.groupby('match_id')['total_seconds'].diff(-1).abs()
    
    # Mask for isolated PFEs: both previous and next differences must be greater than the window
    is_isolated = (df['prev_diff'].fillna(9999) > window_seconds) & \
                  (df['next_diff'].fillna(9999) > window_seconds)
    
    # Keep only isolated PFEs and drop the diff columns
    filtered_df = df[is_isolated].drop(columns=['prev_diff', 'next_diff'])
    
    return filtered_df

In [9]:
def filter_full_window_events(isolated_df, window_seconds=300):
    
    valid_indices = []
    
    # Going through each match to check if the PFEs have a full window before and after within the same half
    for m_id, group in isolated_df.groupby('match_id'):
        events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
        events['total_seconds'] = events['minute'] * 60 + events['second']
        
        # Calculate the time bounds of each period (half) for this match
        period_bounds = events.groupby('period')['total_seconds'].agg(['min', 'max']).to_dict('index')
        
        for idx, row in group.iterrows():
            t = row['total_seconds']
            period = row['period']
            
            if period in period_bounds:
                p_start = period_bounds[period]['min']
                p_end = period_bounds[period]['max']
                
                # Check if there's enough time before and after
                has_full_pre = (t - p_start) >= window_seconds
                has_full_post = (p_end - t) >= window_seconds
                
                if has_full_pre and has_full_post:
                    valid_indices.append(idx)
                    
    # Only return the rows with valid indices
    final_df = isolated_df.loc[valid_indices].copy()
    
    return final_df

In [10]:
def summarize_pfes(pfe_df):

    print(f"Total PFEs detected: {len(pfe_df)}")

    print(f"Number of Smash-and-Grab Goals: {len(pfe_df[pfe_df['event_type'] == 'Smash and Grab'])}")

    print(f"Number of Converted Turnovers: {len(pfe_df[pfe_df['event_type'] == 'Converted Turnover'])}")

    print(f"Number of Crucial Goals: {len(pfe_df[pfe_df['event_type'] == 'Crucial Goal'])}")

    print(f"Number of Let-Offs (Survived Misses): {len(pfe_df[pfe_df['event_type'] == 'Let Off'])}")

    print(f"Number of Final Third Dribbles: {len(pfe_df[pfe_df['event_type'] == 'Final Third Dribble'])}")

In [11]:
all_pfes = []

print(f"Processing {len(match_ids)} matches for PFEs...")

for i, m_id in enumerate(match_ids):
    events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    
    # 1. Detect Smash-and-Grab Goals (PFE-1)
    all_pfes.extend(detect_smash_and_grab(events, m_id))
    
    # 2. Detect Converted Turnovers (PFE-2)
    all_pfes.extend(detect_converted_turnovers(events, m_id))

    # 3. Detect Crucial Goals (PFE-3: Go-Ahead / Equalising)
    all_pfes.extend(detect_crucial_goals(events, m_id))

    # 4. Detect Let-Offs (PFE-4)
    all_pfes.extend(detect_let_offs(events, m_id, 0.30))

    # 5. Detect Final Third Dribbles (PFE-5)
    all_pfes.extend(detect_final_third_dribbles(events, m_id))

    # Progress indicator
    if (i + 1) % 50 == 0:
        print(f"Progress: {i + 1}/{len(match_ids)} matches completed.")

# PFE catalogue
pfe_all_df = pd.DataFrame(all_pfes)
summarize_pfes(pfe_all_df)
pfe_all_df.to_csv('output/pfe/catalogue/events_raw.csv', index=False, encoding='utf-8-sig')

# Isolation
pfe_df = filter_isolated_events(pfe_all_df)
summarize_pfes(pfe_df)
pfe_df = filter_full_window_events(pfe_df)
summarize_pfes(pfe_df)
pfe_df.to_csv('output/pfe/catalogue/events_filtered.csv', index=False, encoding='utf-8-sig')

Processing 1238 matches for PFEs...
Progress: 50/1238 matches completed.
Progress: 100/1238 matches completed.
Progress: 150/1238 matches completed.
Progress: 200/1238 matches completed.
Progress: 250/1238 matches completed.
Progress: 300/1238 matches completed.
Progress: 350/1238 matches completed.
Progress: 400/1238 matches completed.
Progress: 450/1238 matches completed.
Progress: 500/1238 matches completed.
Progress: 550/1238 matches completed.
Progress: 600/1238 matches completed.
Progress: 650/1238 matches completed.
Progress: 700/1238 matches completed.
Progress: 750/1238 matches completed.
Progress: 800/1238 matches completed.
Progress: 850/1238 matches completed.
Progress: 900/1238 matches completed.
Progress: 950/1238 matches completed.
Progress: 1000/1238 matches completed.
Progress: 1050/1238 matches completed.
Progress: 1100/1238 matches completed.
Progress: 1150/1238 matches completed.
Progress: 1200/1238 matches completed.
Total PFEs detected: 11922
Number of Smash-and-G